In [11]:
import importlib
import class_sim.simulation  # Make sure `class_sim` is a Python package.
importlib.reload(class_sim.simulation)  # Reload the latest simulation.py

from class_sim.simulation import SIMULATION  # Re-import SIMULATION

import os
import pandas as pd
import numpy as np
import ctypes
import pickle
import config
importlib.reload(config) 
from config import file_path,geometry_baseline,number_of_timesteps # Separation of parameters required for initialization



* 通过config.py文件，引入初始化所需参数，包括各种文件路径以及baseline(APC10*7)的几何模型、每一次仿真所需总步数（为防止结果无法收敛，可适当增加，基础是1000，至少大于700）
* 初始化后，将自动根据simulation_parameters文件夹内的excel文件生成仿真参数sim文件
* 仿真分为单步仿真和批量仿真
* * 单步仿真接收两种类型参数（1）sim文件的绝对路径 （2）RPM，WIND_SPEED,ANGLE    run_one_simulation(self,RPM,WIND_SPEED,ANGLE,SIM_file_path=None)
* * 批量仿真，会根据excel内的所有工况，针对特定螺旋桨进行全工况仿真 
run_all_simulation(self)
* 改变几何参数，分为三种接口(一旦改变几何参数，会将all_simulation_data进行保存并清零，以便于记录新的螺旋桨几何特性)
* * change_propeller_geometry(self)
* * change_propeller_geometry(self,section_data=None,control_point=numpy) 此时会根据控制点，基于BSpline方法生成几何并替换
* * change_propeller_geometry(self,section_data=numpy,control_point=None) 此时根据提供的22个截面将几何进行替换
  
### Attention 
* 如果在运行期间，想更改仿真工况，需再次运行（如果从头开始运行文件，则无需变动，因为类在初始化过程中自动执行）
* self.generating_simulation_parameters_tuple(self.file_path["simulation_parameter_path"])# get the simulation parameters tuple
* self.generating_sim_file()
  

### initialize the class SIMULATION 

In [13]:
simulation = SIMULATION(file_path=file_path,geometry_baseline=geometry_baseline,device_type='GPU',number_of_timesteps=number_of_timesteps)

delete: /home/gy/Downloads/Propeller_project-main/QBlade_data/QBlade_sim/RPM6000_Wind10_Angle85.sim
delete: /home/gy/Downloads/Propeller_project-main/QBlade_data/QBlade_sim/RPM5000_Wind10_Angle88.sim
delete: /home/gy/Downloads/Propeller_project-main/QBlade_data/QBlade_sim/Base_simulation.qpr
File saved successfully to /home/gy/Downloads/Propeller_project-main/QBlade_data/QBlade_sim/RPM6000_Wind10_Angle85.sim
File saved successfully to /home/gy/Downloads/Propeller_project-main/QBlade_data/QBlade_sim/RPM5000_Wind10_Angle88.sim


### 单步仿真

In [14]:
simulation.run_one_simulation(RPM=5000,WIND_SPEED=11,ANGLE=85)

File saved successfully to /home/gy/Downloads/Propeller_project-main/QBlade_data/QBlade_sim/Base_simulation.sim
Successfully loaded  /home/gy/Downloads/Propeller_project-main/QBladeCE_2.0.8.6/libQBladeCE_2.0.8.6.so.1.0.0


,Time,Thrust,Power,Torque,Thrust_y,Thrust_z,RPM,WIND_SPEED,ANGLE,THRUST,POWER,TORQUE,THRUST_Y,THRUST_Z,Ct,Cp,eta
0,0.073147,-3.899387,-65.509903,-0.104262,0.003107,-0.005299,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011
1,0.073230,-3.877100,-65.290466,-0.103913,-0.005291,0.008630,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011
2,0.073313,-3.883937,-65.308197,-0.103941,-0.002043,0.003149,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011
3,0.073396,-3.859490,-65.039505,-0.103514,-0.004283,0.005864,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011
4,0.073479,-3.918326,-65.710884,-0.104582,0.009914,-0.009908,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,0.082706,-3.887624,-65.378220,-0.104053,-0.001625,0.014494,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011
116,0.082789,-3.891001,-65.427406,-0.104131,0.003538,-0.010568,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011
117,0.082872,-3.832907,-64.756813,-0.103064,0.000405,0.002864,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011
118,0.082955,-3.926677,-65.804337,-0.104731,0.001788,-0.001842,5000.0,11.0,85.0,3.878738,65.265171,0.103873,0.000396,-0.000659,0.109542,87.080426,0.000011


### 批量仿真

In [ ]:
simulation.run_all_simulation()

#### 基于控制点调整几何
当前几何会保存在类属性中

In [16]:
control_points = np.array([0.0198272, 0.03303, 0.01493, 0.00697, 55.05, 19.97, 16.71, 12.54])
simulation.change_propeller_geometry(control_point=control_points)

File saved successfully to '/home/gy/Downloads/Propeller_project-main/QBlade_data/QBlade_sim/Baseline_Blade_Turb/Aero/Baseline_Blade.bld'
File saved successfully to geometry_simulation_dict.pkl


#### 恢复默认几何
当前几何会保存在类属性中

In [ ]:
simualtion.change_propeller_geometry()

#### 基于22个截面调整几何
当前几何会保存在类属性中

In [ ]:
simualtion.change_propeller_geometry(section_data=geometry_baseline)

### 针对某一个螺旋桨跑完所有工况后，如果想保存参数，可以将self.all_simulation_data 添加到一个集合中
* 其中self.all_simulation_data包含螺旋桨几何，针对此类螺旋桨几何对应的各个工况下的仿真数据
* 保存的格式可以是pickle，将自动生成到Simulation_QBlade/class_sim/geometry_simulation_dict.pkl文件中,geometry_num为此类螺旋桨几何的编号,其值为pandas数据框格式包含时间步数（最后120steps），螺旋桨性能参数以及力矩和力参数